In [1]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
from collections import namedtuple
import numpy as np
import torch
#from statsmodels.api import Poisson
import polars as pl
from sklearn.datasets import fetch_openml
from sklearn.linear_model import TweedieRegressor
from actuarial_python_tooling.plots import _prepare_simple_lift_plot_data, plot_simple_lift_plot, plot_obs_pred_by_feature_value, _prepare_obs_pred_by_feature_value
from scipy.optimize import minimize
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import PoissonRegressor
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    KBinsDiscretizer,
    OneHotEncoder,
    StandardScaler,
)
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_tweedie_deviance,
)

In [2]:
from sklearn.datasets import fetch_openml


def load_mtpl2(n_samples=None):
    """Fetch the French Motor Third-Party Liability Claims dataset.

    Parameters
    ----------
    n_samples: int, default=None
      number of samples to select (for faster run time). Full dataset has
      678013 samples.
    """
    # freMTPL2freq dataset from https://www.openml.org/d/41214
    df_freq = fetch_openml(data_id=41214, as_frame=True).data
    df_freq["IDpol"] = df_freq["IDpol"].astype(int)
    df_freq.set_index("IDpol", inplace=True)
    print(df_freq["ClaimNb"].value_counts())

    # freMTPL2sev dataset from https://www.openml.org/d/41215
    df_sev = fetch_openml(data_id=41215, as_frame=True).data

    # sum ClaimAmount over identical IDs
    df_sev = df_sev.groupby("IDpol").sum()

    df = df_freq.join(df_sev, how="left")
    df["ClaimAmount"] = df["ClaimAmount"].fillna(0)

    # unquote string fields
    for column_name in df.columns[df.dtypes.values == object]:
        df[column_name] = df[column_name].str.strip("'")
    return df.iloc[:n_samples]

# Scipy poisson

In [3]:
# https://gist.github.com/ahwillia/40cdbe3b2f2df1806358dd1e6de0743a
n = 1000  # number of datapoints
p = 5  # number of features

# create data
X = 0.3 * np.random.randn(n, p)
true_b = np.random.randn(p)
y = np.random.poisson(np.exp(np.dot(X, true_b)))
yg = np.random.gamma(np.exp(np.dot(X, true_b)))


# LightGBM poisson loss: loss = exp(f) - label * f
# https://github.com/microsoft/LightGBM/blob/master/src/objective/regression_objective.hpp
# SCIKIT LEARN: https://github.com/scikit-learn/scikit-learn/blob/70fdc843a4b8182d97a3508c1a426acc5e87e980/sklearn/_loss/loss.py
#         loss(x_i) = y_true_i * log(y_true_i/exp(raw_prediction_i))
#                    - y_true_i + exp(raw_prediction_i)
# IN JAX: return -1 * jnp.sum(model.y * jnp.log(μ) - μ - jnp.log(jax_factorial(y)))
# IN JAX: return +1 * jnp.sum(-1 * model.y * jnp.log(μ) + μ + jnp.log(jax_factorial(y)))
# der letzte Teil ist unten nicht mit dabei und kann vermutlich entfernt werden?
# wichtig für einige solver! z.B. bei jax for some reason
# loss function and gradient
def f(b):
    Xb = np.dot(X, b)
    exp_Xb = np.exp(Xb)
    loss = exp_Xb.sum() - np.dot(y, Xb)
    grad = np.dot(X.T, exp_Xb - y)
    return loss, grad


# hessian
def hess(b):
    return np.dot(X.T, np.exp(np.dot(X, b))[:, None] * X)


# optimize
result = minimize(f, np.zeros(p), jac=True, hess=hess, method="newton-cg")

print("True regression coeffs: {}".format(true_b))
print("Estimated regression coeffs: {}".format(result.x))

True regression coeffs: [ 0.36520076 -0.01986463  0.78044556  0.81509507  0.19968035]
Estimated regression coeffs: [ 0.22588849 -0.04127814  0.96419791  0.87641083  0.28817466]


In [4]:
def load_mtpl2(n_samples=None):
    """Fetch the French Motor Third-Party Liability Claims dataset.

    Parameters
    ----------
    n_samples: int, default=None
      number of samples to select (for faster run time). Full dataset has
      678013 samples.
    """
    # freMTPL2freq dataset from https://www.openml.org/d/41214
    df_freq = fetch_openml(data_id=41214, as_frame=True).data
    df_freq["IDpol"] = df_freq["IDpol"].astype(int)
    df_freq.set_index("IDpol", inplace=True)
    print(df_freq["ClaimNb"].value_counts())

    # freMTPL2sev dataset from https://www.openml.org/d/41215
    df_sev = fetch_openml(data_id=41215, as_frame=True).data

    # sum ClaimAmount over identical IDs
    df_sev = df_sev.groupby("IDpol").sum()

    df = df_freq.join(df_sev, how="left")
    df["ClaimAmount"] = df["ClaimAmount"].fillna(0)

    # unquote string fields
    for column_name in df.columns[df.dtypes.values == object]:
        df[column_name] = df[column_name].str.strip("'")
    return df.iloc[:n_samples]

In [5]:
df = load_mtpl2()

ClaimNb
0     643953
1      32178
2       1784
3         82
4          7
11         3
5          2
6          1
8          1
16         1
9          1
Name: count, dtype: int64


In [6]:
def score_estimator(
    estimator, X_train, X_test, df_train, df_test, target, weights, tweedie_powers=None, yp=None, yp_test=None
):
    """Evaluate an estimator on train and test sets with different metrics"""

    metrics = [
        ("D² explained", None),  # Use default scorer if it exists
        ("mean abs. error", mean_absolute_error),
        ("mean squared error", mean_squared_error),
    ]
    if tweedie_powers:
        metrics += [
            (
                "mean Tweedie dev p={:.4f}".format(power),
                partial(mean_tweedie_deviance, power=power),
            )
            for power in tweedie_powers
        ]

    res = []
    for subset_label, X, df in [
        ("train", X_train, df_train),
        ("test", X_test, df_test),
    ]:
        y, _weights = df[target], df[weights]
        for score_label, metric in metrics:

            if subset_label == "train" and yp is not None:
                y_pred = yp
            elif subset_label == "test" and yp_test is not None:
                y_pred = yp_test
            elif isinstance(estimator, tuple) and len(estimator) == 2:
                # Score the model consisting of the product of frequency and
                # severity models.
                est_freq, est_sev = estimator
                y_pred = est_freq.predict(X) * est_sev.predict(X)
            else:
                y_pred = estimator.predict(X)

            if metric is None:
                if not hasattr(estimator, "score"):
                    continue
                score = estimator.score(X, y, sample_weight=_weights)
            else:
                print(np.sum(y))
                print(np.sum(y_pred))
                score = metric(y, y_pred, sample_weight=_weights)

            res.append({"subset": subset_label, "metric": score_label, "score": score})

    res = pd.DataFrame(res).set_index(["metric", "subset"]).score.unstack(-1).round(4).loc[:, ["train", "test"]]
    return res

In [7]:
# Correct for unreasonable observations (that might be data error)
# and a few exceptionally large claim amounts
df["ClaimNb"] = df["ClaimNb"].clip(upper=4)
df["Exposure"] = df["Exposure"].clip(upper=1)
df["ClaimAmount"] = df["ClaimAmount"].clip(upper=200000)
# If the claim amount is 0, then we do not count it as a claim. The loss function
# used by the severity model needs strictly positive claim amounts. This way
# frequency and severity are more consistent with each other.
df.loc[(df["ClaimAmount"] == 0) & (df["ClaimNb"] >= 1), "ClaimNb"] = 0

log_scale_transformer = make_pipeline(FunctionTransformer(func=np.log), StandardScaler())

column_trans = ColumnTransformer(
    [
        (
            "binned_numeric",
            KBinsDiscretizer(n_bins=7, random_state=0),
            ["VehAge", "DrivAge"],
        ),
        (
            "onehot_categorical",
            OneHotEncoder(),
            ["VehBrand", "VehPower", "VehGas", "Region", "Area", "BonusMalus"],
        ),
        ("scaled_numeric", StandardScaler(), ["Density"]),
    ],
    remainder="drop",
)
X = column_trans.fit_transform(df)

# Insurances companies are interested in modeling the Pure Premium, that is
# the expected total claim amount per unit of exposure for each policyholder
# in their portfolio:
df["PurePremium"] = df["ClaimAmount"] / df["Exposure"]

# This can be indirectly approximated by a 2-step modeling: the product of the
# Frequency times the average claim amount per claim:
df["Frequency"] = df["ClaimNb"] / df["Exposure"]
df["AvgClaimAmount"] = df["ClaimAmount"] / np.fmax(df["ClaimNb"], 1)

with pd.option_context("display.max_columns", 15):
    print(df[df.ClaimAmount > 0].head())

       ClaimNb  Exposure Area  VehPower  VehAge  DrivAge  BonusMalus VehBrand  \
IDpol                                                                           
139          1      0.75    F         7       1       61          50      B12   
190          1      0.14    B        12       5       50          60      B12   
414          1      0.14    E         4       0       36          85      B12   
424          2      0.62    F        10       0       51         100      B12   
463          1      0.31    A         5       0       45          50      B12   

        VehGas  Density Region  ClaimAmount   PurePremium  Frequency  \
IDpol                                                                  
139    Regular    27000    R11       303.00    404.000000   1.333333   
190     Diesel       56    R25      1981.84  14156.000000   7.142857   
414    Regular     4792    R11      1456.55  10403.928571   7.142857   
424    Regular    27000    R11     10834.00  17474.193548   3.225806   


In [8]:
claim_data = pl.from_pandas(df[df["ClaimAmount"] > 0])
claim_data_pd = df[df["ClaimAmount"] > 0]
sev_data = pl.from_pandas(df)
sev_data_pd = df

In [9]:
df_train, df_test, X_train, X_test = train_test_split(df, X, random_state=0)

In [10]:
glm_freq = PoissonRegressor(alpha=1e-4, solver="newton-cholesky")
glm_freq.fit(X_train, df_train["Frequency"], sample_weight=df_train["Exposure"])

scores = score_estimator(
    glm_freq,
    X_train,
    X_test,
    df_train,
    df_test,
    target="Frequency",
    weights="Exposure",
)
print("Evaluation of PoissonRegressor on target Frequency")
print(scores)

61204.991769744105
39744.90681577586
61204.991769744105
39744.90681577586
18915.708818563355
13275.687336795987
18915.708818563355
13275.687336795987
Evaluation of PoissonRegressor on target Frequency
subset               train    test
metric                            
D² explained        0.0605  0.0574
mean abs. error     0.1367  0.1367
mean squared error  0.2430  0.2229


In [11]:
X_train.shape

(508509, 183)

# Pytorch

In [12]:
X_train.toarray()

array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        , -0.11706596],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.17823721],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.20905585],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        , -0.40175943],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         0.        , -0.43586202],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        , -0.28277928]])

In [13]:
class GLM(torch.nn.Module):
    def __init__(self, input_dim, output_dim):
        super(GLM, self).__init__()
        self.linear = torch.nn.Linear(input_dim, output_dim, dtype=torch.double)

    def forward(self, x):
        out = torch.exp(self.linear(x))
        return out

In [14]:
model = GLM(X_train.shape[1], 1)

In [15]:
criterion = torch.nn.PoissonNLLLoss()

In [16]:
learning_rate = 0.001

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [17]:
# Type of parameter object
print(model.parameters())

# Length of parameters
print(len(list(model.parameters())))

# FC 1 Parameters
print(list(model.parameters())[0].size())

# FC 1 Bias Parameters
print(list(model.parameters())[1].size())

<generator object Module.parameters at 0x138038ba0>
2
torch.Size([1, 183])
torch.Size([1])


In [18]:
batch_size = 2024
int(X_train.shape[0] / batch_size)

251

In [19]:
num_epochs = 8
iter = 0
for epoch in range(num_epochs):
    for i in range(0, int(X_train.shape[0] / batch_size)):
        # Load images as Variable
        batch_x = torch.from_numpy(X_train[i : i + batch_size, :].toarray()).double()
        labels = torch.from_numpy(df_train["Frequency"].to_numpy()[i : i + batch_size]).double()

        # Clear gradients w.r.t. parameters
        optimizer.zero_grad()

        # Forward pass to get output/logits
        outputs = model(batch_x)

        # Calculate Loss: softmax --> cross entropy loss
        loss = criterion(outputs, labels)

        # Getting gradients w.r.t. parameters
        loss.backward()

        # Updating parameters
        optimizer.step()

        iter += 1

        if iter % 500 == 0:
            ...

In [20]:
y_pred = model(torch.from_numpy(X_train.toarray())).detach().numpy()
y_pred_test = model(torch.from_numpy(X_test.toarray())).detach().numpy()

In [21]:
np.sum(y_pred)

67152.42015597816

In [22]:
df_train["Frequency"].sum()

61204.991769744105

In [23]:
scores = score_estimator(
    None, X_train, X_test, df_train, df_test, target="Frequency", weights="Exposure", yp=y_pred, yp_test=y_pred_test
)
print("Evaluation of PoissonRegressor on target Frequency")
print(scores)

61204.991769744105
67152.42015597816
61204.991769744105
67152.42015597816
18915.708818563355
22353.811271165818
18915.708818563355
22353.811271165818
Evaluation of PoissonRegressor on target Frequency
subset               train    test
metric                            
mean abs. error     0.1895  0.1891
mean squared error  0.2485  0.2283


In [24]:
model = GLM(X_train.shape[1], 1)
optimizer = torch.optim.LBFGS(model.parameters(), history_size=3)

In [27]:
num_epochs = 2
iter = 0
for epoch in range(num_epochs):
    for i in range(0, int(X_train.shape[0] / batch_size)):
        # Load images as Variable
        batch_x = torch.from_numpy(X_train[i : i + batch_size, :].toarray()).double()
        labels = torch.from_numpy(df_train["Frequency"].to_numpy()[i : i + batch_size]).double()

        def closure():
            if torch.is_grad_enabled():
                optimizer.zero_grad()
            output = model(batch_x)
            loss = criterion(output, labels)
            if loss.requires_grad:
                loss.backward()
            return loss

        # Clear gradients w.r.t. parameters
        optimizer.step(closure)

        # Forward pass to get output/logits
        outputs = model(batch_x)

        # Calculate Loss: softmax --> cross entropy loss
        loss = criterion(outputs, labels)

        # Getting gradients w.r.t. parameters
        loss.backward()

In [28]:
scores = score_estimator(
    None, X_train, X_test, df_train, df_test, target="Frequency", weights="Exposure", yp=y_pred, yp_test=y_pred_test
)
print("Evaluation of PoissonRegressor on target Frequency")
print(scores)

61204.991769744105
67152.42015597816
61204.991769744105
67152.42015597816
18915.708818563355
22353.811271165818
18915.708818563355
22353.811271165818
Evaluation of PoissonRegressor on target Frequency
subset               train    test
metric                            
mean abs. error     0.1895  0.1891
mean squared error  0.2485  0.2283
